# Lab 1B: SDK Basics / CRUD in Python

**Time**: ~10 min  

In this exercise you will perform Create, Read, Update, and Delete operations on Azure Cosmos DB using the Python `azure-cosmos` SDK v4.x with `DefaultAzureCredential` authentication.

To begin, log in to https://portal.azure.com using your assigned lab account credentials. From the Home page, search for **Azure Cosmos DB** and then open the `cosmoslab***` instance that has been created for you. Also open the `1B_Account_Access.ps1` script file from the current folder. Copy the names of your resource group and account and paste them into the `RESOURCE_GROUP` and `ACCT_NAME` variables in the `1B_Account_Access.ps1` file and then save.

Open a Powershell Terminal to the current folder and run the following command to execute the script to log in to Azure and assign your account access to the data in the Cosmos DB account.

```powershell
.\1B_Account_Access.ps1
```


In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ["COSMOS_ENDPOINT"]
ACCT_NAME = os.environ["COSMOS_ACCOUNT_NAME"]
DB_NAME = "WorkshopData"
CONT_NAME = "Catalog"

print(f"COSMOS_ENDPOINT: {ENDPOINT}")
print(f"Database: {DB_NAME}")
print(f"Container: {CONT_NAME}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey, exceptions
from azure.identity import DefaultAzureCredential

# Create the Cosmos client with DefaultAzureCredential
cred = DefaultAzureCredential()
client = CosmosClient(url=ENDPOINT, credential=cred)

# Get database and container proxies
db = client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to: {ENDPOINT}{DB_NAME}/{CONT_NAME}")

## Cell 5: Create an Item (Create)

Create a new item in the container. The response includes the RU charge.

In [ ]:
import uuid
import json

item_id = str(uuid.uuid4())
item = {
    "id": item_id,
    "name": "Cosmic Item #1",
    "category": "workshop",
    "partitionKey": "workshop",
    "data": {"price": 42.0, "tags": ["cosmos", "demo"]}
}

try:
    response = container.create_item(body=item)
    print(f"Created item {response.get('id')}")
except exceptions.CosmosHttpResponseError as ex:
    if ex.status_code == 409:
        print(f"Item {item_id} already exists")
    else:
        print(f"Error creating item: {ex.message}")
except Exception as ex:
    print(f"Error creating item: {ex}")

## Cell 6: Read an Item (STUDENT EXERCISE)

Read the item you just created. Complete the code below.

**Expected output**: The item JSON with all properties.

**Hint**: Use `container.read_item(id=..., partition_key=...)`. The partition key value is `"workshop"`.

In [ ]:
from azure.cosmos import exceptions as cosmos_exceptions

# Student exercise: get the id from the create response above
read_item_id = item_id  # TODO: set this to the id from the create response
read_partition_key = "workshop"  # TODO: what is the partition key value?

try:
    # TODO: Write the code to read the item
    # read_response = ???
    read_response = container.read_item(item=read_item_id, partition_key=read_partition_key)
    print(f"Read: {read_response}")
except cosmos_exceptions.CosmosResourceNotFoundError:
    print(f"Item {read_item_id} not found")
except Exception as ex:
    print(f"Error reading item: {ex}")

## Cell 7: Upsert the Item (Update)

Update the price of the item using upsert. Upsert will create the item if it does not exist, or replace it if it does.

In [ ]:
item["data"]["price"] = 55.0

try:
    upsert_response = container.upsert_item(body=item)
    print(f"Upserted item {upsert_response.get('id')}")
    print(f"New price: {upsert_response.get('data', {}).get('price')}")
except Exception as ex:
    print(f"Error upserting item: {ex}")

## Cell 8: Delete the Item (Delete)

Delete the item. The response returns HTTP 204 on success.

In [ ]:
try:
    delete_response = container.delete_item(item=item_id, partition_key=read_partition_key)
    print(f"Deleted item {item_id}")
except cosmos_exceptions.CosmosResourceNotFoundError:
    print(f"Item {item_id} not found (may have already been deleted)")
except Exception as ex:
    print(f"Error deleting item: {ex}")

## Lab Complete!

You have completed the CRUD operations exercise. You:
- Connected to Cosmos DB using `DefaultAzureCredential`
- Created an item with `create_item()`
- Read an item with `read_item()`
- Updated an item with `upsert_item()`
- Deleted an item with `delete_item()`